<a href="https://colab.research.google.com/github/busycaesar/Embeddings_And_Cosine_Similarity/blob/Master/Azure/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required dependencies

In [1]:
%pip install langchain_community unstructured azure-identity langchain_openai azure-search-documents


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Import environment variables

In [2]:
from google.colab import userdata
from google.colab import auth

AZURE_OPEN_API_ENDPOINT = userdata.get("AZURE_OPEN_API_ENDPOINT")
AZURE_OPEN_API_KEY = userdata.get("AZURE_OPEN_API_KEY")

VECTOR_SEARCH_ENDPOINT = userdata.get("VECTOR_SEARCH_ENDPOINT")
VECTOR_SEARCH_KEY = userdata.get("VECTOR_SEARCH_KEY")

auth.authenticate_user()

ModuleNotFoundError: No module named 'google'

Fetch the data

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = [
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-900',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-300',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/dp-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-731',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-730',
]

loader = UnstructuredURLLoader(urls)

documents = loader.load()

print(len(documents))

Split the data into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

Store the data into vector database

In [ ]:
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="text-embedding-3-small",
    openai_api_version="2023-05-15",
)

vector_store = AzureSearch(
    azure_search_endpoint=VECTOR_SEARCH_ENDPOINT,
    azure_search_key=VECTOR_SEARCH_KEY,
    index_name='consine-similarity-demo',
    embedding_function=embeddings.embed_query
)

vector_store.add_documents(chunks)

User's query

In [ ]:
user_query = "What resource should I refer to study for AI-900?"

Fetch the relevant chunk of data

In [ ]:
retrieved_docs = vector_store.similarity_search(query=user_query, k=5)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

print(retrieved_docs)

Create prompt template

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
       You are an expert Microsoft Azure cloud instructor and mentor. Your task is to suggest practical project ideas that align with specific Microsoft Azure certification objectives. Base your recommendations strictly on the study guide content provided.

       Study Guide Content: {relevant_chunk_of_data}

       User Question: {prompt}

       Instructions for your response:
        1. Provide 3–5 project ideas relevant to the user’s question and the study guide content.
        2. For each project, include:
          - A clear project title
          - A brief description (2–3 sentences)
          - Which Azure services and tools would be involved
          - How it relates to the certification objectives
        3. Do not include information not covered in the study guide.
        4. Keep your language beginner-friendly and actionable.

       Format your response as a numbered list for clarity.
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="gpt-4.1-mini",
    openai_api_version="2024-12-01-preview",
)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

print(response.content)